# Mean Reversion Strategy

Mean reversion is one of the most well-studied phenomena in quantitative finance. The core hypothesis: asset prices tend to revert toward their historical mean after deviating significantly.

This notebook demonstrates QuantCore's `MeanReversion` strategy, which uses a **z-score** to identify when a price has deviated far enough from its rolling mean to enter a trade.

**Signal logic:**
- z-score < -entry_threshold → BUY (price unusually cheap)
- z-score > +entry_threshold → SELL (price unusually expensive)
- |z-score| < exit_threshold → close position (price has reverted)

---

## Contents
1. Setup & data generation
2. Baseline backtest
3. Performance tearsheet
4. Parameter sensitivity analysis
5. Why it works and when it doesn't

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# add project root so quantcore is importable
sys.path.insert(0, str(Path('..').resolve()))

import quantcore as qc
from quantcore.analytics import (
    calculate_all_metrics,
    calculate_returns,
    rolling_sharpe,
)
from quantcore.plotting import plot_full_tearsheet


def strip_sentinel(results: dict) -> tuple:
    """Strip the zero-timestamp sentinel the engine prepends to the equity curve.

    The C++ engine inserts timestamps_[0] = 0 (Unix epoch) before the first
    bar is processed. Keeping it causes the x-axis to span 1970-present and
    inflates annualisation calculations.
    """
    equity = np.array(results['equity_curve'])
    ts     = np.array(results['timestamps'], dtype='int64')

    # drop every leading entry whose timestamp is zero (the sentinel)
    first_real = np.argmax(ts > 0)
    return equity[first_real:], ts[first_real:]


print(f'QuantCore {qc.version()}')

## 1. Synthetic Data

We simulate an Ornstein-Uhlenbeck process, the continuous-time model that formalises mean reversion. Prices are pulled back toward a long-run mean `mu` with speed `theta`, plus diffusion noise `sigma`.

In [ ]:
np.random.seed(42)

N         = 1000    # trading bars
mu        = 100.0   # long-run mean price
theta     = 0.05    # mean-reversion speed
sigma     = 1.5     # noise magnitude
dt        = 1.0     # one bar per step

# Ornstein-Uhlenbeck simulation
prices = np.empty(N)
prices[0] = mu
for i in range(1, N):
    drift     = theta * (mu - prices[i - 1]) * dt
    diffusion = sigma * np.sqrt(dt) * np.random.randn()
    prices[i] = prices[i - 1] + drift + diffusion

# timestamps in nanoseconds, one bar = one day
start_ns   = int(pd.Timestamp('2022-01-03').value)
day_ns     = int(pd.Timedelta('1D').value)
timestamps = [start_ns + i * day_ns for i in range(N)]

# build BarData objects
bars = [
    qc.BarData(
        'SYNTH',
        timestamps[i],
        prices[i],                   # open
        prices[i] + abs(np.random.randn() * 0.3),  # high
        prices[i] - abs(np.random.randn() * 0.3),  # low
        prices[i],                   # close
        1_000_000.0,
    )
    for i in range(N)
]

fig, ax = plt.subplots(figsize=(14, 4))
dates = pd.to_datetime(timestamps, unit='ns')
ax.plot(dates, prices, linewidth=1.2, color='#2E86AB')
ax.axhline(mu, color='red', linestyle='--', linewidth=1, label=f'Mean = {mu}')
ax.set_title('Simulated Mean-Reverting Price (Ornstein-Uhlenbeck)', fontsize=14, fontweight='bold')
ax.set_ylabel('Price ($)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Bars: {N}')
print(f'Price range: ${prices.min():.2f} – ${prices.max():.2f}')
print(f'Mean: ${prices.mean():.2f}  Std: ${prices.std():.2f}')

## 2. Baseline Backtest

In [ ]:
INITIAL_CAPITAL = 100_000.0

strategy = qc.MeanReversion(lookback=20, entry_threshold=1.5, exit_threshold=0.5)

results = qc.run_backtest(
    strategy=strategy,
    data={'SYNTH': bars},
    initial_capital=INITIAL_CAPITAL,
)

equity_curve, ts = strip_sentinel(results)
returns          = calculate_returns(equity_curve)
metrics          = calculate_all_metrics(equity_curve, risk_free_rate=0.0)

print(metrics)

## 3. Performance Tearsheet

In [ ]:
fig = plot_full_tearsheet(
    equity_curve,
    returns,
    timestamps=ts,
    title='Mean Reversion; Performance Tearsheet',
)
plt.show()

## 4. Parameter Sensitivity

The two most important parameters are `lookback` (how far back we measure the mean/std) and `entry_threshold` (how many standard deviations away from the mean before we trade).

We sweep a grid and measure Sharpe ratio for each combination.

In [ ]:
lookbacks         = [10, 15, 20, 30, 40, 60]
entry_thresholds  = [0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.5]

sharpe_grid = np.full((len(lookbacks), len(entry_thresholds)), np.nan)

for i, lb in enumerate(lookbacks):
    for j, et in enumerate(entry_thresholds):
        # exit threshold must be strictly less than entry threshold
        xt = et * 0.3
        s  = qc.MeanReversion(lookback=lb, entry_threshold=et, exit_threshold=xt)
        r  = qc.run_backtest(strategy=s, data={'SYNTH': bars}, initial_capital=INITIAL_CAPITAL)
        ec, _ = strip_sentinel(r)
        if len(ec) > 1:
            m = calculate_all_metrics(ec, risk_free_rate=0.0)
            # clamp Sharpe to [-3, 3]; values outside this range indicate
            # near-zero volatility (no/few trades) rather than real signal
            sharpe_grid[i, j] = np.clip(m.sharpe_ratio, -3.0, 3.0)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(sharpe_grid, aspect='auto', cmap='RdYlGn', vmin=-1.0, vmax=2.0)

ax.set_xticks(range(len(entry_thresholds)))
ax.set_xticklabels([str(t) for t in entry_thresholds])
ax.set_yticks(range(len(lookbacks)))
ax.set_yticklabels([str(l) for l in lookbacks])
ax.set_xlabel('Entry Threshold (z-score)', fontsize=12)
ax.set_ylabel('Lookback Period (bars)', fontsize=12)
ax.set_title('Sharpe Ratio; Parameter Sensitivity Heatmap', fontsize=14, fontweight='bold')

for i in range(len(lookbacks)):
    for j in range(len(entry_thresholds)):
        if not np.isnan(sharpe_grid[i, j]):
            color = 'white' if abs(sharpe_grid[i, j]) > 1.0 else 'black'
            ax.text(j, i, f'{sharpe_grid[i, j]:.2f}', ha='center', va='center',
                    fontsize=9, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Sharpe Ratio')
plt.tight_layout()
plt.show()

## 5. Why It Works and When It Doesn't

Mean reversion strategies depend critically on the **stationarity** of the underlying price series. We can test this empirically by comparing performance on our mean-reverting OU process versus a random walk (no reversion).

In [ ]:
# random walk, no mean reversion whatsoever
rw_prices    = np.cumsum(np.random.randn(N) * sigma) + mu
rw_bars      = [
    qc.BarData('RW', timestamps[i], rw_prices[i], rw_prices[i] + 0.3,
               rw_prices[i] - 0.3, rw_prices[i], 1_000_000.0)
    for i in range(N)
]

# strong mean-reversion (higher theta)
strong_prices    = np.empty(N)
strong_prices[0] = mu
for i in range(1, N):
    drift              = 0.15 * (mu - strong_prices[i - 1]) * dt
    strong_prices[i]   = strong_prices[i - 1] + drift + sigma * np.sqrt(dt) * np.random.randn()
strong_bars = [
    qc.BarData('STRONG', timestamps[i], strong_prices[i], strong_prices[i] + 0.3,
               strong_prices[i] - 0.3, strong_prices[i], 1_000_000.0)
    for i in range(N)
]

regimes = [
    ('Random Walk (θ=0)',         rw_bars,      '#E84855'),
    ('Weak Reversion (θ=0.05)',   bars,         '#2E86AB'),
    ('Strong Reversion (θ=0.15)', strong_bars,  '#3BB273'),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=False)

for ax, (label, regime_bars, color) in zip(axes, regimes):
    s        = qc.MeanReversion(lookback=20, entry_threshold=1.5, exit_threshold=0.5)
    r        = qc.run_backtest(strategy=s, data={'X': regime_bars}, initial_capital=INITIAL_CAPITAL)
    ec, _    = strip_sentinel(r)
    m        = calculate_all_metrics(ec, risk_free_rate=0.0)
    sharpe   = np.clip(m.sharpe_ratio, -5.0, 5.0)
    ax.plot(ec, linewidth=1.5, color=color)
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_ylabel('Portfolio Value ($)')
    ax.set_xlabel('Bar')
    ax.grid(True, alpha=0.3)
    ax.text(0.05, 0.95, f'Sharpe: {sharpe:.2f}\nReturn: {m.total_return:.1f}%',
            transform=ax.transAxes, fontsize=9, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

fig.suptitle('Mean Reversion Performance Across Market Regimes', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Takeaway: the strategy degrades gracefully as mean-reversion weakens,'
      ' and struggles on pure random walks, as theory predicts.')